# Vibecoderzz AI Ranking Pipeline - Interactive Sandbox

Welcome to the interactive sandbox for our **State-of-the-Art Candidate Ranking Pipeline**. 
This notebook demonstrates how we leverage **Generative AI, LLMs, and Advanced NLP embeddings** to map candidate profiles to a specific Job Description in a highly scalable and robust way.

### Core Features Demonstrated:
1. **Semantic Embeddings**: Using `sentence-transformers` for deep contextual matching.
2. **Heuristic Engine**: Extracted evidence scoring (Career, Production, Availability, Behavioral).
3. **Explainable AI (XAI)**: Generating transparent reasoning for each ranking decision.
4. **Bias Mitigation**: Softened penalty systems ensuring fair evaluation.

In [ ]:
import sys
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer, util
from rich.console import Console
from rich.table import Table

# Add source directory to path so we can import our modules
sys.path.append(os.path.abspath("src"))

try:
    from career_evidence import score_career
    from production_score import score_production
    from behavioral_score import score_behavioral
    from availability_score import score_availability
    from buzzword_penalty import calculate_buzzword_penalty
    from honeypot_filter import detect_honeypot
    from reasoning_generator import generate_reasoning
except ImportError as e:
    print(f"Error importing pipeline modules: {e}")

console = Console()
console.print("[bold green]System Initialized. All modules loaded successfully.[/bold green]")

In [ ]:
# Initialize our advanced embedding model
console.print("[bold blue]Loading SentenceTransformer (all-MiniLM-L6-v2)...[/bold blue]")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Mock Job Description Text
jd_text = """
We are looking for a Senior Machine Learning Engineer with deep expertise in Search, Ranking, 
and Recommendation systems. Must have strong production experience deploying scalable models, 
working with large scale vector databases, and optimizing inference pipelines.
"""
jd_embedding = model.encode(jd_text, normalize_embeddings=True)

# Mock Candidate Data (Simulating parsed JSONL)
candidates = [
    {
        "candidate_id": "c_001",
        "profile": {"years_of_experience": 6},
        "skills": [{"name": "Python"}, {"name": "PyTorch"}, {"name": "Elasticsearch"}, {"name": "Ranking Algorithms"}],
        "career_history": [
            {"title": "Machine Learning Engineer", "description": "Built scalable recommendation systems and search ranking models deployed to 1M+ DAU."}
        ],
        "availability": {"notice_period": "Immediate"},
        "github": {"commits_last_year": 450}
    },
    {
        "candidate_id": "c_002",
        "profile": {"years_of_experience": 2},
        "skills": [{"name": "OpenAI"}, {"name": "ChatGPT"}, {"name": "Generative AI"}, {"name": "Prompt Engineering"}],
        "career_history": [
            {"title": "AI Prompt Intern", "description": "Used Langchain to build wrappers over GPT-4 APIs."}
        ],
        "availability": {"notice_period": "2 Months"},
        "github": {"commits_last_year": 12}
    }
]

console.print(f"[bold green]Loaded mock JD and {len(candidates)} candidates for evaluation.[/bold green]")

## Explainable AI (XAI) Evaluation Engine

Here we run our candidate profiles through the heuristic rules engine and embedding-based semantic matching. We capture every granular signal to provide a completely transparent final score.

In [ ]:
results_table = Table(title="Candidate Evaluation Metrics")
results_table.add_column("Candidate ID", justify="center", style="cyan", no_wrap=True)
results_table.add_column("Career Score", justify="right", style="magenta")
results_table.add_column("Production Score", justify="right", style="magenta")
results_table.add_column("Semantic Match", justify="right", style="green")
results_table.add_column("Buzzword Penalty", justify="right", style="red")
results_table.add_column("Final Pre-Score", justify="right", style="yellow")

detailed_reasonings = {}

for c in candidates:
    # 1. Heuristic Signals
    career_raw = score_career(c)["score"]
    prod_raw = score_production(c)
    behav_multiplier = score_behavioral(c)
    avail_raw = score_availability(c)
    hp_mult = detect_honeypot(c)
    penalty = calculate_buzzword_penalty(c, career_raw)
    
    # 2. Semantic Signals
    skills_text = " ".join([s.get("name", "") for s in c.get("skills", [])])
    exp_text = " ".join([job.get("title", "") + " " + job.get("description", "") for job in c.get("career_history", [])])
    
    skills_emb = model.encode(skills_text, normalize_embeddings=True)
    exp_emb = model.encode(exp_text, normalize_embeddings=True)
    
    sem_raw = float(util.cos_sim(jd_embedding, skills_emb) + util.cos_sim(jd_embedding, exp_emb)) / 2.0
    
    # 3. Base Computation (Simplified for sandbox visualization)
    base_score = (career_raw * 0.4 + prod_raw * 0.2 + avail_raw * 0.1 + sem_raw * 100 * 0.3)
    final_score = (base_score * behav_multiplier * hp_mult) - penalty
    
    results_table.add_row(
        c["candidate_id"], 
        f"{career_raw:.1f}", 
        f"{prod_raw:.1f}", 
        f"{sem_raw:.2f}", 
        f"-{penalty:.1f}", 
        f"{final_score:.1f}"
    )
    
    # Generate XAI Reasoning
    raw_metrics = {
        "candidate_id": c["candidate_id"],
        "career_raw": career_raw,
        "production_raw": prod_raw,
        "behavior_raw": behav_multiplier,
        "availability_raw": avail_raw,
        "semantic_raw": sem_raw,
        "penalty": penalty
    }
    detailed_reasonings[c["candidate_id"]] = generate_reasoning(c, raw_metrics)

console.print(results_table)

## Reasoning Generator Output

The final layer is our XAI Reasoning Generator, which translates these quantitative scores into a human-readable justification for the recruiter or hiring manager.

In [ ]:
for cid, reason in detailed_reasonings.items():
    console.print(f"[bold cyan]Reasoning for {cid}:[/bold cyan]")
    console.print(f"[italic white]{reason}[/italic white]\n")